In [17]:
from ET_pylib.setup import *
setup_notebook()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Shock `numerical` analysis

In [18]:
GEO   = "disaggEA"
print("kernekraft-id'er:", use_geography(GEO))  

yr    = 2040
ids   = NUCLEAR_IDS_BY_GEO[GEO]
RUN   = "disaggEA_2040_kf25om"
SHOCK = "shock_nuc_2040.pkl"

kernekraft-id'er: {'idNUC_DK1': 2035, 'idNUC_DK2': 2035}


In [19]:
baseline_output, shock_output = load_run(RUN, SHOCK)
nuc, cut = nuc_wind_swap(baseline_output, shock_output, yr=yr, nuc_ids=ids)

Kapacitetsbytte, 2040 (GW)
────────────────────────────────────────────────────────────
                                   basis      stød         Δ
────────────────────────────────────────────────────────────
  Kernekraft DK1                    0,00      0,65     +0,65
  Kernekraft DK2                    0,00      0,35     +0,35
  Havvind DK1                       4,50      3,22     -1,28
  Havvind DK2                       1,79      1,25     -0,55
────────────────────────────────────────────────────────────
  Kernekraft i alt                                     +1,00
  Havvind i alt                                        -1,82
  Havvind fjernet pr. GW kernekraft                     1,82
────────────────────────────────────────────────────────────


In [20]:
ctx_el     = build_el_balance_data(baseline_output, shock_output, years=yr)
ctx_el_agg = build_el_balance_data_country(baseline_output, shock_output, years=yr)
shares     = build_supply_shares(ctx_el, aggregates=c_aggs_countries)

## `Printing en masse`

### `2. priser og knaphed`

In [21]:
title = f"Price & scarcity summary ({yr}) "

with pd.option_context("display.width", 200, "display.max_columns", None):
  print("\n" + title + "=" * max(0, 91 - len(title)))
  print(pd.concat(
      [full_summary(ctx_el, ea=e, years=yr) for e in ('DK1', 'DK2')]
    + [full_summary(ctx_el, country="DK", years=yr)]
  ))


Price & scarcity summary (2040) ===========================================================
                    mean     min       max      std   skew  exkurt  nulpristime  mangeltime
region run                                                                                 
DK1    baseline  383.849  40.416  1179.946  277.250  1.191   0.480          0.0     13.6758
       shock     382.816  40.391  1165.536  271.639  1.183   0.471          0.0     13.6758
DK2    baseline  380.953  35.357  1184.470  271.356  1.205   0.542          0.0     15.8676
       shock     378.249  35.378  1157.297  264.729  1.198   0.546          0.0     13.6758
DK     baseline  382.852  38.666  1169.137  274.919  1.195   0.496          0.0     13.6758
       shock     381.244  38.657  1146.154  268.980  1.187   0.492          0.0     13.6758


### `3. Handel`

In [22]:
print(f"\nTrade summary, DK {yr} ─────────────────────────────────")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(full_trade_summary(ctx_el, country="DK", years=yr))

print(f"\nTrade summary, NO {yr} ─────────────────────────────────")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(full_trade_summary(ctx_el, country="NO", years=yr))

print(f"\nTrade summary, SE {yr} ─────────────────────────────────")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(full_trade_summary(ctx_el, country="SE", years=yr))


Trade summary, DK 2040 ─────────────────────────────────
                  export_TWh  import_TWh  net_TWh  export_mDKK  import_mDKK  net_mDKK
partner run                                                                          
DELU    baseline       16.51        4.48    12.03      5362.99      2000.06   3362.93
        shock          16.45        3.93    12.52      5585.00      1472.24   4112.75
GBNIIE  baseline        3.75        4.00    -0.25      1069.64      1467.09   -397.45
        shock           3.82        4.14    -0.32      1135.73      1442.24   -306.51
NL      baseline        1.07        1.50    -0.43       442.15       524.53    -82.38
        shock           1.10        1.59    -0.48       500.72       518.79    -18.07
NO      baseline        1.53        8.79    -7.26       189.96      1688.57  -1498.62
        shock           1.43        8.88    -7.45       178.32      1713.52  -1535.21
SE      baseline        3.03        7.64    -4.61       359.60      2722.78  -2363

In [23]:
print(f"\nTerms of trade per handelspartner, {yr} ─────────────────────────────────")
print_terms_of_trade_partner(ctx_el, country="DK", years=yr)


Terms of trade per handelspartner, 2040 ─────────────────────────────────
Terms of trade, DK per partner — 2040  (X/M in TWh, pX/pM in DKK/MWh, ToT = pX/pM)
            X(b)    X(s)    M(b)    M(s)   pX(b)   pX(s)   pM(b)   pM(s)  ToT(b)  ToT(s)    ΔToT      %Δ
  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  DELU     16.51   16.45    4.48    3.93     325     340     446     375   0.728   0.906  +0.179  +24.56
  GBNIIE    3.75    3.82    4.00    4.14     285     297     367     348   0.778   0.853  +0.076   +9.74
  NL        1.07    1.10    1.50    1.59     413     455     350     326   1.182   1.395  +0.213  +18.06
  NO        1.53    1.43    8.79    8.88     124     125     192     193   0.646   0.646  -0.000   -0.01
  SE        3.03    2.98    7.64    7.66     119     119     356     351   0.333   0.340  +0.007   +1.99
  ──────────────────────────────────────────────────────────────────────────────────────────────────────
  

### `4. Fangstpris og værdifaktor`

In [24]:
title = f"Capture price & value factor, DK ({yr}) "
print("\n" + title + "=" * max(0, 99 - len(title)))
print_capture(capture_summary(ctx_el, country="DK", years=yr), year=yr)


Capture price & value factor, DK (2040) ===========================================================
Capture price (el) — 2040, alle  (GWh / DKK per MWh)
            TWh basis     TWh stød         ΔTWh fangst basis  fangst stød      Δfangst          ΔVF
  ─────────────────────────────────────────────────────────────────────────────────────────────────
  WS        41,012.99    33,035.86    -7,977.13       298.83       299.20         0.37         0.00
  PV        19,006.43    19,030.38        23.95       189.04       187.51        -1.53        -0.00
  WL        13,982.51    13,997.36        14.85       275.48       278.70         3.22         0.01
  BP         1,606.76     1,602.56        -4.20       557.68       550.68        -6.99        -0.01
  ES         1,514.89     1,514.81        -0.08       655.68       645.72        -9.95        -0.02
  BB           436.45       428.42        -8.02       818.46       804.68       -13.78        -0.03
  EX           399.26       389.08       -10.1

### `5. Kapacitetsfaktorer`

In [25]:
print_capacity_factor(capacity_factor_summary(ctx_el, country="DK", years=yr), region="DK", year=yr)

Capacity factor (el, ECap) — 2040, DK  (GWh / pct.)
               basis        stød           Δ    CF basis     CF stød         ΔCF
  ──────────────────────────────────────────────────────────────────────────────
  WS        41,013.0    33,035.9    -7,977.1        50.4        50.5         0.1
  PV        19,006.4    19,030.4        23.9        12.1        12.1         0.0
  WL        13,982.5    13,997.4        14.9        27.8        27.8         0.0
  BP         1,606.8     1,602.6        -4.2        26.9        26.8        -0.1
  ES         1,514.9     1,514.8        -0.1         7.6         7.6        -0.0
  BB           436.4       428.4        -8.0        16.1        15.8        -0.3
  EX           399.3       389.1       -10.2        13.4        13.1        -0.3
  EPE           59.6        57.3        -2.4         4.4         4.3        -0.2
  RO            20.1        20.1         0.0        33.0        33.0         0.0
  CD             0.8         0.7        -0.1         0.0 

c:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\scarcity.py:122: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.26]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mult[is_ex] = (1.0 + m.loc[is_ex, "Cv"] / m.loc[is_ex, "Cm"]).to_numpy()


### `6. Forbrugeroverskud`

In [26]:
print_cs_table(baseline_output, shock_output, year=yr)

ΔCS 2040 — million DKK (electricity)

                         ΔCS       price    quantity
────────────────────────────────────────────────────
Denmark
  DK1                   44.1        44.0         0.1
  DK2                   58.3        58.3         0.1
  DK3                    0.0         0.0         0.0
  Denmark total        102.4       102.3         0.1
Norway
  NOS                  -62.5       -62.5         0.0
  NON                   -7.2        -7.2         0.0
  NOM                  -12.9       -12.9         0.0
  Norway total         -82.6       -82.6         0.0
Sweden
  SE1                   -3.7        -3.7         0.0
  SE2                   -6.7        -6.7         0.0
  SE3                  -28.4       -28.4         0.0
  SE4                   14.2        14.2         0.0
  Sweden total         -24.6       -24.6         0.0
════════════════════════════════════════════════════
Total                   -4.8        -4.9         0.1


### `7. Producentoverskud`

In [27]:
areas = c_aggs["DK"]
maps = dict(id2Pt=baseline_output["id2Pt"].vals, id2EA=shock_output["id2EA"].vals)
dec_maps = dict(**maps,
                id2HA=shock_output["id2HA"].vals,
                id2cIdxE=shock_output["id2cIdxE"].vals,
                id2cIdxH=shock_output["id2cIdxH"].vals)

bt = producer_profit(baseline_output, shock_output, areas=areas, year=yr,
                     split_nuclear=True, **maps)
dec = profit_decomposition(baseline_output, shock_output, areas=areas, year=yr,
                           split_nuclear=True, **dec_maps)
print_producer_profit(bt, year=yr, areas=areas, decomp=dec)

Producer profit — 2040, DK1+DK2+DK3  (million DKK)
                base       shock     Δprofit        pris      mængde        rest
  ──────────────────────────────────────────────────────────────────────────────
  WS         6,962.5     5,630.4    -1,332.0        91.3       -11.6    -1,411.7
  PV         3,134.9     3,099.5       -35.4       -36.6         0.1         1.1
  ES           510.9       487.8       -23.2       -23.7         0.9        -0.3
  BP           205.9       197.4        -8.5        -9.8         0.2         1.1
  EX           126.3       118.8        -7.5        -7.5         0.1        -0.0
  BB          -182.4      -188.0        -5.5        -6.7         0.1         1.1
  EPE           11.8        11.0        -0.8        -0.9         0.0         0.1
  HS         1,924.7     1,924.0        -0.7        -0.7         0.1        -0.1
  SH           154.1       153.9        -0.3        -0.3         0.0         0.0
  RO             3.5         3.5        -0.0        -0.0  

### `8. Produktion`

In [28]:
by_qty = producer_generation(baseline_output, shock_output, areas=areas, year=yr,
                             split_nuclear=True, **maps)
print_producer_qty(by_qty, year=yr, areas=areas)

Generation — 2040, DK1+DK2+DK3  (TWh)
                base       shock        ΔTWh
  ──────────────────────────────────────────
  WS          41.013      33.036      -7.977
  EX           0.399       0.389      -0.010
  BB           0.436       0.428      -0.008
  BP           1.607       1.603      -0.004
  EPE          0.060       0.057      -0.002
  CD           0.001       0.001      -0.000
  RO           0.020       0.020       0.000
  ES           4.200       4.209       0.010
  WL          13.983      13.997       0.015
  PV          26.031      26.058       0.027
  Nuclear       0.000       8.013       8.013
  ──────────────────────────────────────────
  Total       87.749      87.811       0.062



### `9. Flaskehalsindtægt`

In [29]:
for label, db in (("Baseline", baseline_output), ("Shock", shock_output)):
    zones, countries = ci_by_country(db, yr)
    print(f"{label}\n\n--- CI, DK (mio. DKK) ---")
    print(zones[zones["land"] == "DK"])
    print(countries.loc[["DK"]])

Baseline

--- CI, DK (mio. DKK) ---
     CI_mioDKK land
DK1     2060.3   DK
DK2      402.2   DK
DK3      185.6   DK
      CI_mioDKK
land           
DK       2648.1
Shock

--- CI, DK (mio. DKK) ---
     CI_mioDKK land
DK1     2015.1   DK
DK2      386.6   DK
DK3      182.1   DK
      CI_mioDKK
land           
DK       2583.8


### `10. Velfærd`

*fleksibilitetskorrigering ved*
$$
    \Delta W = \sum_{h\in\mathcal H} (p_h-MC-s) \cdot \Delta q_h
$$

In [30]:
nuc_flex_value(shock_output, year=2040).round(2)               # must-run
nuc_flex_value(shock_output, year=2040, u_min=0.3).round(2)    # minimumslast 50 %

,TWh,dTWh,p_extra,dW
id,,,,
idNUC_DK1,5.20,0.10,62.18,-7.63
idNUC_DK2,2.81,0.05,56.92,-4.24
total,8.01,0.16,60.38,-11.87


*samlede velfærdsgevinster*

In [31]:
blocks = welfare_block_by_rate(
    baseline_output, shock_output,
    {"FM": "fm", "4 pct.": 0.04, "6 pct.": 0.06},
    year=yr, regions=("DK",), markets=("E",),
)
print_welfare_blocks(blocks, regions="DK")

Velfærdseffekt, DK — 2040
(mio. kr./år)
─────────────────────────────────────────────────────────────────────
                                           FM      4 pct.      6 pct.
─────────────────────────────────────────────────────────────────────
  Δ Forbrugeroverskud                 +102,40     +102,40     +102,40
  Δ Producentoverskud                 +113,83     +113,83     +113,83
  Δ Flaskehalsindtægter                -64,30      -64,30      -64,30
  Annuitet, netto                     -654,80     -885,98   -1.677,82
  Systemomkostning, VE              +1.150,68   +1.150,68   +1.150,68
  Systemomkostning, kernekraft        -245,03     -245,03     -245,03
─────────────────────────────────────────────────────────────────────
  Netto velfærdseffekt                +402,79     +171,60     -620,24
─────────────────────────────────────────────────────────────────────
  heraf annuitet (+ = omkostning, − = sparet):
    Kernekraft (ny)                 +2.504,38   +2.853,22   +4.149,17
   

### `11. LCOE`

In [32]:
lcoe_table(shock_output, year=yr, r='fm').round(1)

c:\Users\LinusLindquist\e-tank\E-Tank - Data\PROJEKTER\04 KERNEKRAFT\Grøn Reform beregninger og følgegruppe\GrønREFORM_2.0\KF26_wrapper\ET_pylib\annuity.py:729: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  el_fac = (1 + _lcoe_yr(db, "Cv", year).div(_lcoe_yr(db, "Cm", year))).reindex(ids).fillna(1.0)


,CF,annuitet,FOM,MC,LCOE
tech,,,,,
KK,0.9,31.3,12.5,10.7,54.4
WS,0.5,22.9,9.0,4.0,35.9
WL,0.3,21.6,5.0,1.5,28.2
PV,0.1,19.2,7.9,0.0,27.0
VE (vægtet),0.2,21.3,7.8,2.1,31.3
